# File Handling in Python

Working with files is an essential part of scientific programming and data analysis. In this notebook, you will learn the fundamentals of file handling, including understanding file paths, downloading data files, and advanced user interaction through graphical file dialogs.

## Learning Objectives

By the end of this notebook, you will be able to:
- Distinguish between relative and absolute paths
- Construct cross-platform file paths using `os.sep.join()`
- Download files programmatically for different environments (Colab vs local)
- Use `pooch` for robust data file management
- Create simple file selection interfaces with `tkinter`
- Implement a "recent file" memory system

## 1. Understanding File Paths

File paths are the addresses that tell your computer where to find specific files. Understanding the difference between relative and absolute paths is crucial for writing portable code.

### Absolute vs Relative Paths

- **Absolute path:** Starts from the root directory of your file system
  - Linux/macOS: `/home/user/documents/data.csv`
  - Windows: `C:\\Users\\user\\Documents\\data.csv`
  
- **Relative path:** Starts from your current working directory
  - `data/myfile.csv` (looks for a 'data' folder in current directory)
  - `../data/myfile.csv` (goes up one directory, then into 'data' folder)

### Why This Matters

Relative paths make your code more portable - it will work on different computers and operating systems without modification, as long as the relative structure is maintained.

In [ ]:
import os

# Check your current working directory
print("Current working directory:", os.getcwd())

# Example: absolute path to a file
abs_path = os.path.abspath("example.txt")
print("Absolute path to 'example.txt':", abs_path)

# Example: relative path from current directory
rel_path = os.path.relpath(abs_path)
print("Relative path from current directory:", rel_path)

## 2. Cross-Platform Path Construction

Different operating systems use different path separators:
- Windows uses backslashes: `\\` that need to put double into a string, as the first backslash is understood as symbol
- Linux/macOS use forward slashes: `/`

Python's `os` module provides `os.sep` which automatically uses the correct separator for your system. The `os.sep.join()` method constructs paths that work on any operating system.

### Why Use `os.sep.join()`?

This method ensures your code works correctly regardless of the operating system, making it more robust and portable.

In [ ]:
# Examine the path separator for your system
print("Path separator for this system:", repr(os.sep))

# Construct a cross-platform path
folder = "Data"
filename = "experimental_results.csv"
full_path = os.sep.join([os.getcwd(), folder, filename])
print("Cross-platform path:", full_path)

# Alternative: using os.path.join() (more commonly used)
alt_path = os.path.join(os.getcwd(), folder, filename)
print("Alternative path construction:", alt_path)
print("Are they the same?", full_path == alt_path)

## 3. Environment-Aware File Downloading

When working with Jupyter notebooks, you often need to handle different environments:
- **Google Colab:** A cloud-based environment where you might need to download files from repositories
- **Local environment:** Where you might have files stored locally

The following pattern is commonly used to detect the environment and handle file access appropriately.

In [ ]:
import sys
import os

# Check if we're running in Google Colab
if "google.colab" in sys.modules:
    print("Running in Google Colab - setting up repository download...")
    
    # Set path for Colab environment
    path_to_files = os.sep.join([os.getcwd(), "BioChemistry", "projects", "notebook7"])
    
    # Download the repository (example - replace with actual repository)
    !git clone https://github.com/luchem/BioChemistry.git --depth=1
    
else:
    print("Running in local environment - using local data directory...")
    
    # Set path for local environment
    path_to_files = os.sep.join([os.getcwd(), "Data"])

print(f"Files will be accessed from: {path_to_files}")

The idea of this beeing that if you are on colab the !git command should download the whole directory from Github.

You then need to specify where in the filestructure created the data is located. If you are working locally, we assume that the data is already on the computer (otherwise simply copy the git command).

In both cased the you need to know where the files are. This is stored in the variable "path_to_files".

# Task:

 Check in your system that the files are indeed there!

## 4. Modern File Downloading with Pooch

[Pooch](https://www.fatiando.org/pooch/) is a Python library designed to simplify downloading and caching data files. It's particularly useful for scientific applications where you need to:
- Download data files from the internet
- Cache files locally to avoid repeated downloads
- Verify file integrity with checksums
- Handle different file formats and compression

### Why Use Pooch?

- **Automatic caching:** Files are downloaded once and cached locally
- **Integrity checking:** Verifies files haven't been corrupted
- **Cross-platform:** Works on all operating systems
- **Flexible:** Supports various data sources and formats

In [ ]:
# Install pooch if not already available
!pip install -q pooch

import pooch

# Example: Download a file using pooch
# Note: Replace with actual URLs and known hashes for real applications
url = "https://raw.githubusercontent.com/luchem/KEMM30/master/lectures/Data/CabChamp_0.csv"

# Download file (pooch will cache it automatically)
fname = pooch.retrieve(
        url=url,
        known_hash=None,  # In production, you should specify a hash for verification
        path=path_to_files  # Download to our data directory
    )


### Task: Understanding Pooch

1. Run the cell above and observe the output
2. Run it again - notice how pooch uses the cached version on the second run
3. Check the `path_to_files` directory to see where the file was downloaded
   use "with open(fname) as f:
           print(f.readline())
       "
   to check if you can read the file

# Advanced: File Selection with Tkinter

Sometimes you need users to select files interactively. Python's built-in `tkinter` library provides a simple file dialog for this purpose. We'll also implement a "recent file" system that remembers the last selected file.

### How It Works

1. Check if a "recent" file exists and load the stored filename
2. Use the recent file by default, or open a file dialog if needed
3. Save the selected filename for future use

This pattern is useful for applications where users frequently work with the same files.

In [ ]:
import tkinter as tk
from tkinter import filedialog
import os

# Configuration
recent_filename = "recent_file.txt"  # File to store the recent selection
open_gui = False  # Set to True to force the file dialog to open

def get_recent_file():
    """Get the most recently used file path from disk."""
    try:
        with open(recent_filename, "r") as f:
            recent_path = f.read().strip()
            # Verify the file still exists
            if os.path.exists(recent_path):
                return recent_path
            else:
                print(f"Recent file no longer exists: {recent_path}")
                return None
    except FileNotFoundError:
        print("No recent file found.")
        return None

def save_recent_file(filename):
    """Save the filename as the most recent selection."""
    with open(recent_filename, "w") as f:
        f.write(filename)
    print(f"Saved '{filename}' as recent file.")

def select_file_with_dialog():
    """Open a file selection dialog."""
    # Create a root window and hide it
    root = tk.Tk()
    root.withdraw()  # Hide the main window
    
    # Open file dialog
    filename = filedialog.askopenfilename(
        title="Select a data file",
        filetypes=[
            ("All files", "*.*"),
            ("CSV files", "*.csv"),
            ("Text files", "*.txt"),
            ("Python files", "*.py")
        ]
    )
    
    # Clean up the root window
    root.destroy()
    
    return filename if filename else None

# Main file selection logic
print("=== File Selection System ===")

# Try to get recent file first
selected_file = get_recent_file()

# Open dialog if no recent file or if explicitly requested
if selected_file is None or open_gui:
    print("Opening file selection dialog...")
    selected_file = select_file_with_dialog()
    
    # Save the selection if a file was chosen
    if selected_file:
        save_recent_file(selected_file)
else:
    print(f"Using recent file: {selected_file}")

# Display result
if selected_file:
    print(f"\nSelected file: {selected_file}")
    
    # Show some basic file information
    try:
        size = os.path.getsize(selected_file)
        print(f"File size: {size} bytes")
        print(f"File extension: {os.path.splitext(selected_file)[1]}")
    except Exception as e:
        print(f"Could not access file info: {e}")
else:
    print("No file selected.")

### Task: Testing the File Selection System

1. Run the cell above - it should either use a recent file or open a dialog
2. Change `open_gui = True` and run again to force the dialog
3. Select a file and observe how it's saved as the recent file
4. Run the cell again with `open_gui = False` to see the recent file being used

**Note:** The file dialog might not work in some online environments like Colab, but it will work in local Jupyter installations.

## 6. Reading and Processing Files

Once you have a file path, you'll often need to read and process the file contents. Here's a robust approach that handles potential errors gracefully.

In [ ]:
def read_file_safely(filepath):
    """Read a file with error handling."""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read()
        return content
    except FileNotFoundError:
        print(f"Error: File not found - {filepath}")
        return None
    except PermissionError:
        print(f"Error: Permission denied - {filepath}")
        return None
    except UnicodeDecodeError:
        print(f"Error: Cannot decode file - {filepath} (try different encoding)")
        return None
    except Exception as e:
        print(f"Unexpected error reading {filepath}: {e}")
        return None

# Example: Create a small test file and read it
test_file = os.sep.join([path_to_files, "test_data.txt"])

# Create test file
try:
    with open(test_file, 'w') as f:
        f.write("# Sample Data File\n")
        f.write("# Created for file handling demonstration\n")
        f.write("Temperature,Pressure,Volume\n")
        f.write("298.15,1.013,22.4\n")
        f.write("273.15,1.000,22.4\n")
    print(f"Created test file: {test_file}")
    
    # Read and display the file
    content = read_file_safely(test_file)
    if content:
        print("\nFile contents:")
        print(content)
        
except Exception as e:
    print(f"Error creating test file: {e}")

## 7. Working with Multiple Files

In research, you often need to process multiple files in a directory. Here's how to handle this systematically using the patterns you've learned.

In [ ]:
def process_directory_files(directory_path, file_extension=".txt"):
    """Process all files with a given extension in a directory."""
    file_data = {}  # Dictionary to store file contents
    
    if not os.path.exists(directory_path):
        print(f"Directory does not exist: {directory_path}")
        return file_data
    
    # Get list of files in directory
    try:
        files = os.listdir(directory_path)
        print(f"Found {len(files)} items in directory")
    except Exception as e:
        print(f"Error reading directory: {e}")
        return file_data
    
    # Process each file
    for filename in files:
        if filename.endswith(file_extension):
            filepath = os.sep.join([directory_path, filename])
            
            print(f"Processing: {filename}")
            content = read_file_safely(filepath)
            
            if content is not None:
                file_data[filename] = {
                    'content': content,
                    'size': len(content),
                    'lines': len(content.splitlines())
                }
                print(f"  ✓ Read {len(content)} characters, {len(content.splitlines())} lines")
            else:
                print(f"  ✗ Failed to read file")
    
    return file_data

# Example: Process files in our data directory
print("=== Processing Directory Files ===")
results = process_directory_files(path_to_files, ".txt")

print(f"\nSuccessfully processed {len(results)} files:")
for filename, info in results.items():
    print(f"  {filename}: {info['size']} characters, {info['lines']} lines")

## Summary

In this notebook, you've learned essential file handling concepts:

1. **Path Management**: Understanding absolute vs relative paths and using `os.sep.join()` for cross-platform compatibility

2. **Environment Detection**: Automatically adapting your code for different environments (Colab vs local)

3. **Modern File Downloading**: Using `pooch` for robust, cached file downloads

4. **User Interaction**: Creating file selection dialogs with `tkinter` and implementing a recent file system

5. **Error Handling**: Reading files safely with proper exception handling

6. **Batch Processing**: Working with multiple files systematically

### Best Practices

- Always use relative paths when possible for portability
- Use `os.sep.join()` or `os.path.join()` for cross-platform path construction
- Implement proper error handling for file operations
- Cache downloaded files to avoid repeated downloads
- Provide fallbacks for different environments

### Next Steps

You can now apply these concepts to:
- Load experimental data from various sources
- Create robust data processing pipelines
- Build user-friendly interfaces for file selection
- Handle different file formats (CSV, JSON, etc.) with appropriate libraries